Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0

In [ ]:
# @title TIPS Zero-Shot Segmentation notebook

#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at

#  https://www.apache.org/licenses/LICENSE-2.0

#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

# Setup

## Install Dependencies

In [ ]:
# @title Install dependencies and clone TIPS repo
import dataclasses
import json
import math
import os
import shutil
import subprocess
import sys
import warnings
import zipfile
import gdown
from typing import Callable

import numpy as np
import PIL.Image
import torch
import torch.nn.functional as F
import torchvision.transforms as TVT
import torchvision.transforms.functional as TVTF
import tqdm
from torch import Tensor, nn

# Root directory for all files (Colab default is /content).
ROOT_DIR = os.getcwd()
TIPS_DIR = os.path.join(ROOT_DIR, 'tips')

# Install required packages.
!pip install -q torch torchvision torchaudio
!pip install -q tensorflow_text mediapy jax jaxlib scikit-learn

# Clone the TIPS repository.
if not os.path.exists(TIPS_DIR):
  !git clone https://github.com/google-deepmind/tips.git {TIPS_DIR}

# Add the root directory to PYTHONPATH so that `tips.*` imports work.
if ROOT_DIR not in sys.path:
  sys.path.insert(0, ROOT_DIR)

print(f'ROOT_DIR: {ROOT_DIR}')
print(f'TIPS_DIR: {TIPS_DIR}')
print('Installation complete!')

## Download and Extract Data

In [ ]:
# @title Download the checkpoints, sample images, and ADE20k Dataset
import urllib.request

CHECKPOINT_BASE_URL = 'https://storage.googleapis.com/tips_data/v2_0/checkpoints/pytorch'
TOKENIZER_URL = 'https://storage.googleapis.com/tips_data/v1_0/checkpoints/tokenizer.model'
IMAGE_BASE_URL = 'https://raw.githubusercontent.com/google-deepmind/tips/main/scenic/images'
ADE20K_URL = 'http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip'
ADE20K_TMP_PATH = '/content/ADEChallengeData2016.zip'

# Directories for checkpoints and images (under ROOT_DIR).
CKPT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
IMG_DIR = os.path.join(ROOT_DIR, 'images')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# Download checkpoints for selected variant.
vision_ckpt_name, text_ckpt_name = 'tips_v2_oss_l14_vision.npz', 'tips_v2_oss_l14_text.npz'

for ckpt_name in [vision_ckpt_name, text_ckpt_name]:
  ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
  if not os.path.exists(ckpt_path):
    print(f'\nDownloading {ckpt_name}...')
    urllib.request.urlretrieve(f'{CHECKPOINT_BASE_URL}/{ckpt_name}', ckpt_path)
    print(f'  Saved to {ckpt_path}')
  else:
    print(f'  {ckpt_name} already exists.')

# Download tokenizer.
tokenizer_file = os.path.join(CKPT_DIR, 'tokenizer.model')
if not os.path.exists(tokenizer_file):
  print('\nDownloading tokenizer...')
  urllib.request.urlretrieve(TOKENIZER_URL, tokenizer_file)
  print(f'  Saved to {tokenizer_file}')
else:
  print('  tokenizer.model already exists.')

# Download sample images.
sample_images = ['example_image.jpg', 'example_image_2.jpg']
for img_name in sample_images:
  img_path = os.path.join(IMG_DIR, img_name)
  if not os.path.exists(img_path):
    print(f'\nDownloading {img_name}...')
    urllib.request.urlretrieve(f'{IMAGE_BASE_URL}/{img_name}', img_path)
    print(f'  Saved to {img_path}')
  else:
    print(f'  {img_name} already exists.')

# Download ADE20K Dataset
if not os.path.exists(ADE20K_TMP_PATH):
  print(f'\nDownloading ADEChallengeData2016.zip...')
  urllib.request.urlretrieve(ADE20K_URL, ADE20K_TMP_PATH)
  print(f'  Saved to {ADE20K_TMP_PATH}')
else:
  print('  ADEChallengeData2016.zip already exists.')

print('\nAll downloads complete!')

In [ ]:
#@title Extract the ADE20K Dataset

ADE20K_DIR = "/tmp/ADEChallengeData2016"

if not os.path.isdir(ADE20K_DIR):
    zip_path = "/content/ADEChallengeData2016.zip"
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/tmp/")
    print(f"Extracted to {ADE20K_DIR}")
else:
    print(f"ADE20K already at {ADE20K_DIR}")

ADE20K_CLASS_NAMES = (
    "wall", "building", "sky", "floor", "tree", "ceiling", "road",
    "bed", "windowpane", "grass", "cabinet", "sidewalk", "person", "earth",
    "door", "table", "mountain", "plant", "curtain", "chair", "car", "water",
    "painting", "sofa", "shelf", "house", "sea", "mirror", "rug", "field",
    "armchair", "seat", "fence", "desk", "rock", "wardrobe", "lamp", "bathtub",
    "railing", "cushion", "base", "box", "column", "signboard",
    "chest of drawers", "counter", "sand", "sink", "skyscraper", "fireplace",
    "refrigerator", "grandstand", "path", "stairs", "runway", "case",
    "pool table", "pillow", "screen door", "stairway", "river", "bridge",
    "bookcase", "blind", "coffee table", "toilet", "flower", "book", "hill",
    "bench", "countertop", "stove", "palm", "kitchen island", "computer",
    "swivel chair", "boat", "bar", "arcade machine", "hovel", "bus", "towel",
    "light", "truck", "tower", "chandelier", "awning", "streetlight", "booth",
    "television receiver", "airplane", "dirt track", "apparel", "pole", "land",
    "bannister", "escalator", "ottoman", "bottle", "buffet", "poster", "stage",
    "van", "ship", "fountain", "conveyer belt", "canopy", "washer", "plaything",
    "swimming pool", "stool", "barrel", "basket", "waterfall", "tent", "bag",
    "minibike", "cradle", "oven", "ball", "food", "step", "tank", "trade name",
    "microwave", "pot", "animal", "bicycle", "lake", "dishwasher", "screen",
    "blanket", "sculpture", "hood", "sconce", "vase", "traffic light", "tray",
    "ashcan", "fan", "pier", "crt screen", "plate", "monitor", "bulletin board",
    "shower", "radiator", "glass", "clock", "flag",
)

NORMALIZE_TIPS = TVT.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])


class Ade20kDataset(torch.utils.data.Dataset):
    CLASS_NAMES = ADE20K_CLASS_NAMES

    def __init__(self, root: str, split: str, transform: Callable) -> None:
        self.transform = transform
        img_dir = os.path.join(root, "images", split)
        ann_dir = os.path.join(root, "annotations", split)
        self.images = sorted([
            os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(".jpg")
        ])
        self.annotations = sorted([
            os.path.join(ann_dir, f) for f in os.listdir(ann_dir) if f.endswith(".png")
        ])
        assert len(self.images) == len(self.annotations), (
            f"Mismatch: {len(self.images)} images vs {len(self.annotations)} annotations"
        )
        print(f"Loaded {len(self.images)} ADE20K {split} images.")

    def __getitem__(self, idx: int) -> tuple:
        img = PIL.Image.open(self.images[idx]).convert("RGB")
        mask = np.array(PIL.Image.open(self.annotations[idx]))
        img = self.transform(img)
        mask = torch.from_numpy(mask).long()
        mask = torch.where((mask == 0) | (mask == 255), 255, mask - 1)
        return img, mask

    def __len__(self) -> int:
        return len(self.images)

## Define Functions and Configure Data

In [ ]:
#@title Define Segmentation Helper Functions — Value Attention

#######
# The key difference: `encode_image_value_attention` extracts the
# **raw values** (V) from the last transformer block's attention layer,
# projected through the output projection, instead of the standard
# softmax-weighted attention output.
# This is the MaskCLIP approach used in Dense Features / ViTWithValues.

def _get_all_blocks(model_image):
    if model_image.chunked_blocks:
        blocks = []
        for chunk in model_image.blocks:
            for blk in chunk:
                if not isinstance(blk, nn.Identity):
                    blocks.append(blk)
        return blocks
    return list(model_image.blocks)


def encode_image_value_attention(model_image, img: Tensor) -> Tensor:
    B, _, H, W = img.shape
    P = model_image.patch_size
    new_H = math.ceil(H / P) * P
    new_W = math.ceil(W / P) * P

    if (H, W) != (new_H, new_W):
        img = F.interpolate(img, size=(new_H, new_W), mode="bicubic", align_corners=False)

    B, _, h_i, w_i = img.shape

    x = model_image.prepare_tokens_with_masks(img)

    num_register = model_image.num_register_tokens
    all_blocks = _get_all_blocks(model_image)
    for i, blk in enumerate(all_blocks):
        if i < len(all_blocks) - 1:
            x = blk(x)
        else:
            x_normed = blk.norm1(x)
            b_dim, n_dim, c_dim = x_normed.shape
            qkv = (
                blk.attn.qkv(x_normed)
                .reshape(b_dim, n_dim, 3, blk.attn.num_heads, c_dim // blk.attn.num_heads)
                .permute(2, 0, 3, 1, 4)
            )
            v = qkv[2]
            v_out = v.transpose(1, 2).reshape(b_dim, n_dim, c_dim)
            v_out = blk.attn.proj(v_out)
            v_out = blk.ls1(v_out)
            x_val = v_out + x

            y_val = blk.norm2(x_val)
            y_val = blk.ls2(blk.mlp(y_val))
            x_val = x_val + y_val

    x_val = model_image.norm(x_val)
    patch_tokens = x_val[:, 1 + num_register:, :]
    blocks_patches = patch_tokens.reshape(B, h_i // P, w_i // P, -1).contiguous()
    return blocks_patches


class ShortSideResize(nn.Module):
    def __init__(self, size: int, interpolation: TVT.InterpolationMode) -> None:
        super().__init__()
        self.size = size
        self.interpolation = interpolation

    def forward(self, img: Tensor) -> Tensor:
        _, h, w = TVTF.get_dimensions(img)
        if (w <= h and w == self.size) or (h <= w and h == self.size):
            return img
        if w < h:
            new_w = self.size
            new_h = int(self.size * h / w)
            return TVTF.resize(img, [new_h, new_w], self.interpolation)
        else:
            new_h = self.size
            new_w = int(self.size * w / h)
            return TVTF.resize(img, [new_h, new_w], self.interpolation)


def predict_whole(model_image, img: Tensor, text_features: Tensor) -> Tensor:
    _, H, W = img.shape
    blocks_feats = encode_image_value_attention(model_image, img.unsqueeze(0))
    _, h, w, _ = blocks_feats.shape
    blocks_feats = blocks_feats.squeeze(0)

    blocks_feats = F.normalize(blocks_feats, p=2, dim=-1)
    cos = torch.einsum("cd,hwd->chw", text_features, blocks_feats)

    return cos

def predict_slide(model_image, img: Tensor, text_features: Tensor, side: int, stride: int) -> Tensor:
    _, H, W = img.shape
    num_classes, _ = text_features.shape
    probs = torch.zeros([num_classes, H, W], device="cuda")
    counts = torch.zeros([H, W], device="cuda")
    h_grids = max(H - side + stride - 1, 0) // stride + 1
    w_grids = max(W - side + stride - 1, 0) // stride + 1
    for i in range(h_grids):
        for j in range(w_grids):
            y1 = i * stride
            x1 = j * stride
            y2 = min(y1 + side, H)
            x2 = min(x1 + side, W)
            y1 = max(y2 - side, 0)
            x1 = max(x2 - side, 0)

            img_window = img[:, y1:y2, x1:x2]
            cos = predict_whole(model_image, img_window, text_features)

            cos = F.interpolate(
                cos.unsqueeze(0),
                size=img_window.shape[1:],
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)
            probs[:, y1:y2, x1:x2] += cos.softmax(dim=0)
            counts[y1:y2, x1:x2] += 1
    probs /= counts

    return probs

In [ ]:
# @title Configure Prompt Templates
# Subset of ImageNet prompts used in the TCL paper.
_SUB_IMAGENET_PROMPTS = [
    'itap of a {}.',
    'a bad photo of a {}.',
    'a origami {}.',
    'a photo of the large {}.',
    'a {} in a video game.',
    'art of the {}.',
    'a photo of the small {}.',
]

# Templates used in the TCL paper.
# https://github.com/khanrc/tcl/blob/main/datasets/templates.py#L145
_TCL_PROMPTS = _SUB_IMAGENET_PROMPTS + [
    'a photo of many {}.',
    'a photo of {}s.',
]

PROMPT_TEMPLATES = _TCL_PROMPTS
print(f"Using {len(PROMPT_TEMPLATES)} TCL prompt templates")

## Prepare Model

In [ ]:
#@title Import and Patch TIPS Model

import glob
import io
import os
import mediapy as media
import numpy as np
from PIL import Image
import tensorflow as tf
import tensorflow_text
from tips.pytorch import image_encoder
from tips.pytorch import text_encoder
from tips.scenic.utils import feature_viz
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms

IMAGE_MEAN = (0, 0, 0)
IMAGE_STD = (1.0, 1.0, 1.0)
PATCH_SIZE = 14
MAX_LEN = 64
VOCAB_SIZE = 32000
MAX_SEQ_LEN = 64

device = "cuda"

# Load the tokenizer directly from the downloaded checkpoint
tokenizer_path = '/content/checkpoints/tokenizer.model'
with tf.io.gfile.GFile(tokenizer_path, 'rb') as f:
    tokenizer = tensorflow_text.SentencepieceTokenizer(f.read())

def load_image_bytes(file_name):
  with open(file_name, 'rb') as fd:
    image_bytes = io.BytesIO(fd.read())
    pil_image = Image.open(image_bytes)
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ])
    input_tensor = transform(pil_image)
    input_batch = input_tensor.unsqueeze(0)

  return input_batch

def get_pretokenized_batch(class_idx: int) -> tuple[torch.Tensor, torch.Tensor]:
    # Generate texts on the fly using ADE20K_CLASS_NAMES and PROMPT_TEMPLATES
    class_name = ADE20K_CLASS_NAMES[class_idx]
    texts = [template.format(class_name) for template in PROMPT_TEMPLATES]

    # Tokenize the texts
    tokens = tokenizer.tokenize(texts).to_list()

    max_l = min(max(len(ids) for ids in tokens), MAX_SEQ_LEN)
    num = len(tokens)
    token_ids = np.zeros((num, max_l), dtype=np.int64)
    paddings = np.ones((num, max_l), dtype=np.float32)

    for i, ids in enumerate(tokens):
        length = min(len(ids), max_l)
        token_ids[i, :length] = ids[:length]
        paddings[i, :length] = 0.0

    return (
        torch.from_numpy(token_ids).to(device, non_blocking=True),
        torch.from_numpy(paddings).to(device, non_blocking=True),
    )

# Monkey-patch to address the device mismatch in TextEncoder
def new_text_encoder_call(self, ids, paddings):
    """Applies TextEncoder module with device-aware positional embeddings."""
    _, seq_length = ids.shape
    mask = (paddings == 0).type(torch.float32)
    mask = mask.permute(1, 0)  # NL -> LN
    x = self.token_embedding(ids)
    if self.scale_sqrt_depth:
      x = x * (self.embedding_dim**0.5)

    pos_embeddings = self.pos_embedder(seq_length=seq_length)
    x = x + pos_embeddings.to(x.device)

    x = x.permute(1, 0, 2)  # NLD -> LND
    x = self.transformer(x, mask)
    x = x.permute(1, 0, 2)  # LND -> NLD
    x = self.ln_final(x)
    x = self.pooling(x, compatible_paddings=paddings[:, :, None])
    return x

text_encoder.TextEncoder.__call__ = new_text_encoder_call
print("TextEncoder has been patched and tokenizer is loaded for on-the-fly tokenization!")

In [ ]:
#@title Configure the TIPS model.
# Set the input image shape and variant.
image_size = 448  # @param {type: "number"}
stride = 336 # @param {type: "number"}
side = 448  # @param {type: "number"}
resize = 448  # @param {type: "number"}
mode = "slide"  # @param {type: "string"}
# variant is already set in the download cell above.

# Checkpoint and tokenizer paths (absolute paths).
image_encoder_checkpoint = os.path.join(CKPT_DIR, vision_ckpt_name)
text_encoder_checkpoint = os.path.join(CKPT_DIR, text_ckpt_name)
tokenizer_path = os.path.join(CKPT_DIR, 'tokenizer.model')

print(f'Image encoder checkpoint: {image_encoder_checkpoint}')
print(f'Text encoder checkpoint: {text_encoder_checkpoint}')
print(f'Tokenizer path: {tokenizer_path}')

In [ ]:
#@title Load TIPS Model

## Load Vision Model
weights_image = dict(np.load(image_encoder_checkpoint, allow_pickle=False))
for key in weights_image:
  weights_image[key] = torch.tensor(weights_image[key])
# ffn_layer = 'swiglu' if variant == 'g' else 'mlp'
ffn_layer = 'mlp'

embeddings_image, spatial_features = [], []

with torch.no_grad():
  # Load the vision encoder.
  model_image = image_encoder.vit_large(
      img_size=image_size,
      patch_size=PATCH_SIZE,
      ffn_layer=ffn_layer,
      block_chunks=0,
      init_values=1.0,
      interpolate_antialias=True,
      interpolate_offset=0.0,
  )
  model_image.load_state_dict(weights_image)
  model_image = model_image.to(device)

## Load Text Model
text_config = {
    "hidden_size": 1024,
    "mlp_dim": 4096,
    "num_heads": 16,
    "num_layers": 12,
}

with open(text_encoder_checkpoint, 'rb') as fin:
  inbuffer = io.BytesIO(fin.read())
np_weights_text = dict(np.load(inbuffer, allow_pickle=False))

weights_text = {}
for key, value in np_weights_text.items():
  weights_text[key] = torch.from_numpy(value)

temperature = weights_text.pop('temperature')
with torch.no_grad():
  # Load the text encoder.
  model_text = text_encoder.TextEncoder(
      text_config,
      vocab_size=VOCAB_SIZE,
  )

  model_text.load_state_dict(weights_text)
  model_text = model_text.to(device)

# Run Inference

In [ ]:
# @title Compute mIoU
def compute_miou(confusion_matrix: torch.Tensor) -> float:
    intersection = confusion_matrix.diag()
    union = confusion_matrix.sum(dim=1) + confusion_matrix.sum(dim=0) - intersection
    iou = intersection / (union + 1e-10)
    valid = union > 0
    return iou[valid].mean().item()

transform = TVT.Compose(
    [
        ShortSideResize(resize, TVT.InterpolationMode.BICUBIC),
        TVT.ToTensor(),
        NORMALIZE_TIPS,
    ]
)
dataset = Ade20kDataset(ADE20K_DIR, "validation", transform)
class_names = dataset.CLASS_NAMES
num_classes = len(class_names)
print(f"Dataset: {len(dataset)} images, {num_classes} classes")
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=None,
    num_workers=0,
    shuffle=False,
    pin_memory=True,
)

text_feats = []
with torch.no_grad():
    for ci, class_name in enumerate(tqdm.tqdm(class_names, desc="Class names", unit="name", ncols=0)):
        ids, paddings = get_pretokenized_batch(ci)
        ids, paddings = ids.to(device), paddings.to(device)
        feats = model_text(ids, paddings)
        feats = F.normalize(feats, p=2, dim=-1)
        feats = feats.mean(dim=0)
        feats = F.normalize(feats, p=2, dim=-1)
        text_feats.append(feats)
text_feats = torch.stack(text_feats).float()
print(f"Text features shape: {text_feats.shape}, dtype: {text_feats.dtype}")

confusion_matrix = torch.zeros(num_classes, num_classes, dtype=torch.long, device=device)
for idx, (img, target) in enumerate(tqdm.tqdm(dataloader, desc="Segmentation", unit="img", ncols=0)):
    _, H, W = img.shape
    H_target, W_target = target.shape
    img = img.to(device, dtype=torch.float, non_blocking=True)
    target = target.to(device, non_blocking=True)
    if idx == 0:
        tqdm.tqdm.write(f"Image shape:  {img.shape}")
        tqdm.tqdm.write(f"Target shape: {target.shape}")

    with torch.inference_mode():
        if mode == "whole":
            pred = predict_whole(model_image, img, text_feats)
        elif mode == "slide":
            pred = predict_slide(model_image, img, text_feats, side, stride)
        else:
            raise ValueError(f"Unknown mode {mode}")

    pred = F.interpolate(pred.unsqueeze(0), size=(H_target, W_target), mode="bilinear", align_corners=False)
    pred = pred.squeeze(0).argmax(dim=0)

    mask = target != 255
    pred_valid = pred[mask]
    target_valid = target[mask]
    if pred_valid.numel() > 0:
        indices = target_valid * num_classes + pred_valid
        confusion_matrix += torch.bincount(indices, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

miou = compute_miou(confusion_matrix)
print(f"Segmentation mIoU: {100 * miou:.2f}")